<a href="https://colab.research.google.com/github/leonardo2004/Calculo-Numerico/blob/main/Matriz_Gauss/Matriz_Gauss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NOME: Leonardo Tomasela Leal RA: 170291

In [17]:
"""Animação da Eliminação de Gauss em GIF.

Uso:  python gauss.py
Requer apenas: numpy, matplotlib, Pillow.
"""
import io

import numpy as np
import matplotlib

matplotlib.use("Agg")  # nenhuma janela é aberta
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from PIL import Image

EPS = 1e-9

# Paleta
C_FUNDO = "#0f172a"
C_PAINEL = "#1e293b"
C_TEXTO = "#e5e7eb"
C_APAGADO = "#64748b"
C_PIVO = "#fbbf24"
C_ALVO = "#f87171"
C_OPER = "#22d3ee"
C_TITULO = "#f8fafc"
C_SOL = "#4ade80"

SUB = str.maketrans("0123456789", "₀₁₂₃₄₅₆₇₈₉")


# ----------------------------------------------------------------------
# 1. Coleta de dados
# ----------------------------------------------------------------------
def coletar_dados():
    while True:
        try:
            n = int(input("Dimensão do sistema (n): ").strip())
            if n < 2 or n > 8:
                raise ValueError
            break
        except ValueError:
            print("  Digite um inteiro entre 2 e 8.")

    print(f"Digite cada linha de [A | b] com {n + 1} valores separados por espaço.")
    A = []
    for i in range(n):
        while True:
            try:
                valores = [float(v.replace(",", ".")) for v in input(f"Linha {i + 1}: ").split()]
                if len(valores) != n + 1:
                    raise ValueError
                A.append(valores)
                break
            except ValueError:
                print(f"  Entrada inválida: informe exatamente {n + 1} números.")

    while True:
        r = input("Pivoteamento parcial? (s/n): ").strip().lower()
        if r in ("s", "n"):
            break
        print("  Responda com 's' ou 'n'.")
    return n, np.array(A, dtype=float), r == "s"


# ----------------------------------------------------------------------
# 2. Algoritmo com histórico
# ----------------------------------------------------------------------
def _fmt(v):
    s = f"{0.0 if abs(v) < EPS else v:.2f}"
    if s == "-0.00":
        s = "0.00"
    return s.replace("-", "−")


def gauss_com_historico(A, pivotear):
    M = np.array(A, dtype=float)
    n = M.shape[0]
    passos = []

    def registrar(desc, pivo=None, alvo=None):
        snap = M.copy()
        snap[np.abs(snap) < EPS] = 0.0
        passos.append({"matriz": snap, "desc": desc, "pivo": pivo, "alvo": alvo})

    registrar("Matriz aumentada original")

    r = 0
    for c in range(n):
        if r >= n:
            break
        if pivotear:
            p = max(range(r, n), key=lambda i: abs(M[i, c]))
            if abs(M[p, c]) <= EPS:
                p = None
        else:
            p = r if abs(M[r, c]) > EPS else next(
                (i for i in range(r + 1, n) if abs(M[i, c]) > EPS), None)

        if p is None:
            registrar(f"Coluna {c + 1} sem pivô — seguindo para a próxima coluna")
            continue

        if p != r:
            M[[r, p]] = M[[p, r]]
            motivo = "pivoteamento parcial" if pivotear else "pivô nulo"
            registrar(f"L{r + 1} ↔ L{p + 1}   ({motivo})", pivo=(r, c), alvo=p)

        for i in range(r + 1, n):
            if abs(M[i, c]) <= EPS:
                continue
            m = M[i, c] / M[r, c]
            M[i] = M[i] - m * M[r]
            M[i, c] = 0.0
            registrar(f"L{i + 1} ← L{i + 1} − ({_fmt(m)})·L{r + 1}", pivo=(r, c), alvo=i)
        r += 1

    # Classificação
    tipo, solucao = "SPD", None
    for i in range(n):
        if np.all(np.abs(M[i, :n]) < EPS) and abs(M[i, n]) > EPS:
            tipo = "SI"
            break
    else:
        if r < n:
            tipo = "SPI"

    if tipo == "SPD":
        x = np.zeros(n)
        for i in range(n - 1, -1, -1):
            x[i] = (M[i, n] - M[i, i + 1:n] @ x[i + 1:]) / M[i, i]
        solucao = x

    final = M.copy()
    final[np.abs(final) < EPS] = 0.0
    passos.append({"matriz": final, "desc": "Sistema escalonado", "pivo": None, "alvo": None})
    return passos, solucao, tipo


# ----------------------------------------------------------------------
# 3. Renderização de um frame
# ----------------------------------------------------------------------
def desenhar_frame(matriz, descricao, pivo, linha_alvo, n, titulo="Eliminação de Gauss",
                   rodape=None, cor_rodape=C_SOL, apagado=False, contador=None):
    W, H = 10, 5.625  # 16:9
    fig = plt.figure(figsize=(W, H), facecolor=C_FUNDO)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.set_facecolor(C_FUNDO)
    T = ax.transAxes

    ax.text(0.5, 0.91, titulo, transform=T, ha="center", va="center", color=C_TITULO,
            fontsize=22, fontweight="bold", family="sans-serif")
    if contador:
        ax.text(0.95, 0.94, contador, transform=T, ha="right", va="center",
                color=C_APAGADO, fontsize=11, family="sans-serif")

    # Geometria da matriz
    rowh = min(0.10, 0.48 / n)
    cy = 0.56
    alt = n * rowh
    gap = 0.04
    largura_total = 0.62
    w = (largura_total - gap) / (n + 1)
    x0 = 0.5 - largura_total / 2
    xs = [x0 + (j + 0.5) * w for j in range(n)] + [x0 + (n + 0.5) * w + gap]
    x_sep = x0 + n * w + gap / 2
    fs = max(9, min(20, int(0.85 * w * W * 72 / 3.7)))

    # Painel
    pad_x, pad_y = 0.085, 0.045
    painel = FancyBboxPatch(
        (x0 - pad_x, cy - alt / 2 - pad_y), largura_total + 2 * pad_x, alt + 2 * pad_y,
        boxstyle="round,pad=0,rounding_size=0.03", transform=T,
        facecolor=C_PAINEL, edgecolor="#334155", linewidth=1.5)
    ax.add_patch(painel)

    # Colchetes grandes
    fs_col = alt * H * 72 / 1.15
    xb_e, xb_d = x0 - 0.035, x0 + largura_total + 0.035
    ax.text(xb_e, cy, "[", transform=T, ha="center", va="center", color=C_TEXTO,
            fontsize=fs_col, fontweight="bold", family="monospace")
    ax.text(xb_d, cy, "]", transform=T, ha="center", va="center", color=C_TEXTO,
            fontsize=fs_col, fontweight="bold", family="monospace")

    # Separador vertical
    ax.plot([x_sep, x_sep], [cy - alt / 2 + 0.008, cy + alt / 2 - 0.008], transform=T,
            color=C_APAGADO, linewidth=2, solid_capstyle="round")

    # Células
    for i in range(n):
        y = cy + alt / 2 - (i + 0.5) * rowh
        ax.text(x0 - pad_x + 0.015, y, f"L{i + 1}", transform=T, ha="left", va="center",
                color=C_APAGADO, fontsize=11, family="sans-serif")
        for j in range(n + 1):
            cor, peso = C_TEXTO, "normal"
            if apagado:
                cor = C_APAGADO
            else:
                if linha_alvo is not None and i == linha_alvo:
                    cor = C_ALVO
                if pivo is not None and (i, j) == tuple(pivo):
                    cor, peso = C_PIVO, "bold"
            ax.text(xs[j], y, _fmt(matriz[i][j]), transform=T, ha="center", va="center",
                    color=cor, fontsize=fs, fontweight=peso, family="monospace")

    # Rótulo da operação
    y_base = cy - alt / 2 - pad_y
    ax.text(0.5, y_base - 0.075, descricao, transform=T, ha="center", va="center",
            color=C_OPER, fontsize=17, style="italic", family="sans-serif")

    if rodape:
        ax.text(0.5, max(0.05, y_base - 0.165), rodape, transform=T, ha="center", va="center",
                color=cor_rodape, fontsize=19, fontweight="bold", family="monospace")
    return fig


# ----------------------------------------------------------------------
# 4. Geração do GIF
# ----------------------------------------------------------------------
def _fig_para_imagem(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=120, facecolor=fig.get_facecolor())
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf).convert("RGB")


def gerar_gif(passos, solucao, tipo, n, nome="gauss.gif"):
    frames, duracoes = [], []
    total = len(passos)

    # Fade-in simples: matriz em cinza, depois colorida
    p0 = passos[0]
    frames.append(_fig_para_imagem(desenhar_frame(p0["matriz"], "", None, None, n, apagado=True)))
    duracoes.append(400)

    for k, p in enumerate(passos[:-1], start=1):
        fig = desenhar_frame(p["matriz"], p["desc"], p["pivo"], p["alvo"], n,
                             contador=f"Passo {k}/{total - 1}")
        frames.append(_fig_para_imagem(fig))
        duracoes.append(900)

    ult = passos[-1]
    if tipo == "SPD":
        txt = "   ".join(f"x{str(i + 1).translate(SUB)} = {_fmt(v)}" for i, v in enumerate(solucao))
        desc, cor = "Substituição regressiva → sistema possível e determinado (SPD)", C_SOL
    elif tipo == "SI":
        txt = "Sistema impossível (SI): não existe solução."
        desc, cor = "Linha 0 = b com b ≠ 0 detectada", C_ALVO
    else:
        txt = "Sistema possível e indeterminado (SPI): infinitas soluções."
        desc, cor = "Há variáveis livres (posto < n)", C_PIVO
    fig = desenhar_frame(ult["matriz"], desc, None, None, n, rodape=txt, cor_rodape=cor)
    frames.append(_fig_para_imagem(fig))
    duracoes.append(3000)

    frames[0].save(nome, save_all=True, append_images=frames[1:], duration=duracoes, loop=0)


if __name__ == "__main__":
    n, A, pivotear = coletar_dados()
    passos, solucao, tipo = gauss_com_historico(A, pivotear)
    gerar_gif(passos, solucao, tipo, n)
    print(f"Tipo do sistema: {tipo}")
    if solucao is not None:
        print("Solução:", ", ".join(f"x{i + 1} = {v:.2f}" for i, v in enumerate(solucao)))
    print("GIF gerado: gauss.gif")

Dimensão do sistema (n): 3
Digite cada linha de [A | b] com 4 valores separados por espaço.
Linha 1: 1 1 1 1
Linha 2: 2 2 2 2 
Linha 3: 4 4 4 4
Pivoteamento parcial? (s/n): s
Tipo do sistema: SPI
GIF gerado: gauss.gif
